# 使用链式调用实现推理模型的结构化输出

最初发布（2024 年 9 月）的 [o1](https://openai.com/index/introducing-openai-o1-preview/) 推理模型具备先进的能力，但不支持 [结构化输出](https://platform.openai.com/docs/guides/structured-outputs/examples)。

这意味着对 o1 的请求无法获得可靠的类型安全，只能依靠提示词本身来返回可用的 JSON。

在本指南中，我们将探讨两种方法，在使用 OpenAI API 时提示 o1 模型（具体为 `o1-preview`）返回有效的 JSON 格式。

# 提示

让 `o1-preview` 返回 JSON 响应的最简单方式是在提示中明确要求。

我们来看一个示例：
- 获取一个列出公司的 wikipedia 页面
- 判断哪些公司最能从 AI 能力中获益
- 以 JSON 格式返回结果，以便其他系统接入使用

In [11]:
import requests
from openai import OpenAI

client = OpenAI()

def fetch_html(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return None

url = "https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue"
html_content = fetch_html(url)

json_format = """
{
    companies: [
        {
            \"company_name\": \"OpenAI\",
            \"page_link\": \"https://en.wikipedia.org/wiki/OpenAI\",
            \"reason\": \"OpenAI would benefit because they are an AI company...\"
        }
    ]
}
"""

o1_response = client.chat.completions.create(
    model="o1-preview",
    messages=[
        {
            "role": "user", 
            "content": f"""
You are a business analyst designed to understand how AI technology could be used across large corporations.

- Read the following html and return which companies would benefit from using AI technology: {html_content}.
- Rank these propects by opportunity by comparing them and show me the top 3. Return only as a JSON with the following format: {json_format}"
"""
        }
    ]
)

print(o1_response.choices[0].message.content)

{
    "companies": [
        {
            "company_name": "Walmart",
            "page_link": "https://en.wikipedia.org/wiki/Walmart",
            "reason": "Walmart could benefit from AI technology by enhancing their supply chain management, optimizing inventory levels, improving customer service through AI-powered chatbots, and providing personalized shopping experiences. AI can help Walmart forecast demand more accurately, reduce operational costs, and increase overall efficiency."
        },
        {
            "company_name": "UnitedHealth Group",
            "page_link": "https://en.wikipedia.org/wiki/UnitedHealth_Group",
            "reason": "UnitedHealth Group could leverage AI technology to improve patient care through predictive analytics, personalize treatment plans, detect fraudulent claims, and streamline administrative processes. AI can assist in early disease detection, improve diagnostic accuracy, and enhance data analysis for better health outcomes."
        },
   

可以看到，响应已经相当不错——它返回了包含合适内容的 JSON。然而，它同样会遇到现有仅靠提示推断 JSON 的用例所面临的问题：
- 你必须手动将此 JSON 处理为类型安全的结构
- 模型拒绝（refusal）不会[作为单独的结构由 API 显式返回](https://platform.openai.com/docs/guides/structured-outputs/refusals)

# 结构化输出

现在我们使用 [结构化输出](https://platform.openai.com/docs/guides/structured-outputs) 来完成这项任务。为了启用此功能，我们会将 `o1-preview` 的响应与对 `gpt-4o-mini` 的后续请求串联起来，后者可以有效处理初始 o1-preview 响应返回的数据。

In [12]:
from pydantic import BaseModel
from devtools import pprint

class CompanyData(BaseModel):
    company_name: str
    page_link: str
    reason: str

class CompaniesData(BaseModel):
    companies: list[CompanyData]

o1_response = client.chat.completions.create(
    model="o1-preview",
    messages=[
        {
            "role": "user", 
            "content": f"""
You are a business analyst designed to understand how AI technology could be used across large corporations.

- Read the following html and return which companies would benefit from using AI technology: {html_content}.
- Rank these propects by opportunity by comparing them and show me the top 3. Return each with {CompanyData.__fields__.keys()}
"""
        }
    ]
)

o1_response_content = o1_response.choices[0].message.content

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user", 
            "content": f"""
Given the following data, format it with the given response format: {o1_response_content}
"""
        }
    ],
    response_format=CompaniesData,
)

pprint(response.choices[0].message.parsed)

CompaniesData(
    companies=[
        CompanyData(
            company_name='Walmart',
            page_link='https://en.wikipedia.org/wiki/Walmart',
            reason=(
                'As the largest retailer, Walmart can significantly benefit from AI by optimizing supply chain and inv'
                'entory management, improving demand forecasting, personalizing customer experiences, and enhancing in'
                '-store operations through AI-driven analytics.'
            ),
        ),
        CompanyData(
            company_name='JPMorgan Chase',
            page_link='https://en.wikipedia.org/wiki/JPMorgan_Chase',
            reason=(
                'As a leading financial institution, JPMorgan Chase can leverage AI for fraud detection, risk manageme'
                'nt, personalized banking services, algorithmic trading, and enhancing customer service with AI-powere'
                'd chatbots and virtual assistants.'
            ),
        ),
        CompanyData(
  

# 结论

结构化输出让你的代码具备可靠的类型安全，并简化提示。此外，它还能让你复用对象 schema，更轻松地集成到现有工作流中。

o1 系列模型目前不支持结构化输出，但我们可以通过串联两个请求，复用 `gpt-4o-mini` 现有的结构化输出功能。这一流程目前需要两次调用，但与 `o1-preview`/`o1-mini` 的调用相比，第二次 `gpt-4o-mini` 调用的成本应该微乎其微。